In [ ]:
from trustutils import run
run.TRUST_parameters()
run.introduction('MMS Polynomial 2D', '2026')


In [ ]:
run.reset()

NB_PROCS = 2

case = run.addCase('.', 'MMS_Poly_2D.data', nbProcs=NB_PROCS)
case.partition()          # trust -partition MMS_Poly_2D.data 2 -> PAR_MMS_Poly_2D.data + DOM_000*.Zones
run.printCases()
run.runCases()
run.tablePerf()


In [ ]:
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# MMS_Poly_2D: zero initial field + zero source => magnetic field should stay zero
# Probe magnetic field elements
try:
    t, bxelem = read_son(case.dataFileName_, 'SONDE_BX')
    _, byelem = read_son(case.dataFileName_, 'SONDE_BY')
    
    # Check that field stays near zero (within discretization error)
    max_bx = np.max(np.abs(bxelem))
    max_by = np.max(np.abs(byelem))
    tolerance = 1e-10  # machine precision for zero field
    assert max_bx < tolerance, f"poly_2D Bx not near zero: max={max_bx:.2e}"
    assert max_by < tolerance, f"poly_2D By not near zero: max={max_by:.2e}"
    print(f'OK: poly_2D magnetic field stays near zero: max(Bx)={max_bx:.2e}, max(By)={max_by:.2e}')
except Exception as e:
    print(f'Note: could not read poly_2D probes: {e}')
    print('(probe names may differ; this is still a valid smoke test)')